# Traditional Response Modeling Baseline

This notebook trains ordinary response models to predict customer visits. These models serve as a baseline for comparison with uplift models. Traditional response models predict who will respond (visit), but they do not explicitly model the causal effect of treatment. This means they may recommend targeting customers who would have responded anyway, even without receiving the email.

### Reproducibility Note

This notebook uses the shared train/test split and feature matrices created in `02_feature_matrix.ipynb`. All paths, random seeds, and feature lists are loaded from `src/config.py`. Model predictions and results are saved to standardized output directories so they can be compared against uplift models in later notebooks.

In [ ]:
import sys
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Allows notebook to import from src/ when running inside notebooks/
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    PROCESSED_DATA_DIR,
    PREDICTIONS_DIR,
    MODEL_RESULTS_DIR,
    FIGURES_DIR,
    RANDOM_STATE,
    PRIMARY_OUTCOME,
)

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Predictions output:", PREDICTIONS_DIR)
print("Model results output:", MODEL_RESULTS_DIR)

### Load preprocessed data

In [ ]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")

y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTrain visit rate:", y_train.mean())
print("Test visit rate:", y_test.mean())

X_train.head()

### Train logistic regression baseline

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    solver='lbfgs'
)

lr_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")
print("Number of features:", len(lr_model.coef_[0]))

### Generate predictions

In [ ]:
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]
y_pred_lr = (y_pred_proba_lr >= 0.5).astype(int)

print("Predicted probabilities (first 10):")
print(y_pred_proba_lr[:10])
print("\nPredicted classes (first 10):")
print(y_pred_lr[:10])

### Evaluate logistic regression

In [ ]:
lr_auc = roc_auc_score(y_test, y_pred_proba_lr)
lr_precision = precision_score(y_test, y_pred_lr, zero_division=0)
lr_recall = recall_score(y_test, y_pred_lr, zero_division=0)
lr_f1 = f1_score(y_test, y_pred_lr, zero_division=0)
lr_brier = brier_score_loss(y_test, y_pred_proba_lr)

print("Logistic Regression Performance")
print("=" * 40)
print(f"ROC AUC:    {lr_auc:.4f}")
print(f"Precision:  {lr_precision:.4f}")
print(f"Recall:     {lr_recall:.4f}")
print(f"F1 Score:   {lr_f1:.4f}")
print(f"Brier Score: {lr_brier:.4f}")

print("\nConfusion Matrix:")
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

### Train XGBoost model

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully.")

### Generate XGBoost predictions

In [ ]:
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_pred_proba_xgb >= 0.5).astype(int)

print("Predicted probabilities (first 10):")
print(y_pred_proba_xgb[:10])

### Evaluate XGBoost

In [ ]:
xgb_auc = roc_auc_score(y_test, y_pred_proba_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb, zero_division=0)
xgb_recall = recall_score(y_test, y_pred_xgb, zero_division=0)
xgb_f1 = f1_score(y_test, y_pred_xgb, zero_division=0)
xgb_brier = brier_score_loss(y_test, y_pred_proba_xgb)

print("XGBoost Performance")
print("=" * 40)
print(f"ROC AUC:    {xgb_auc:.4f}")
print(f"Precision:  {xgb_precision:.4f}")
print(f"Recall:     {xgb_recall:.4f}")
print(f"F1 Score:   {xgb_f1:.4f}")
print(f"Brier Score: {xgb_brier:.4f}")

print("\nConfusion Matrix:")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

### Model comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'ROC AUC': [lr_auc, xgb_auc],
    'Precision': [lr_precision, xgb_precision],
    'Recall': [lr_recall, xgb_recall],
    'F1 Score': [lr_f1, xgb_f1],
    'Brier Score': [lr_brier, xgb_brier]
})

print("\nModel Comparison")
print("=" * 80)
display(comparison)

### ROC curves

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.3f})', linewidth=2)
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {xgb_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Response Models', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "response_baseline_roc.png", dpi=150)
plt.show()

print("ROC curve saved to:", FIGURES_DIR / "response_baseline_roc.png")

### Calibration curves

In [ ]:
prob_true_lr, prob_pred_lr = calibration_curve(y_test, y_pred_proba_lr, n_bins=10, strategy='uniform')
prob_true_xgb, prob_pred_xgb = calibration_curve(y_test, y_pred_proba_xgb, n_bins=10, strategy='uniform')

plt.figure(figsize=(8, 6))
plt.plot(prob_pred_lr, prob_true_lr, marker='o', label='Logistic Regression', linewidth=2)
plt.plot(prob_pred_xgb, prob_true_xgb, marker='s', label='XGBoost', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=1)
plt.xlabel('Mean Predicted Probability', fontsize=12)
plt.ylabel('Fraction of Positives', fontsize=12)
plt.title('Calibration Curve - Response Models', fontsize=14)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "response_baseline_calibration.png", dpi=150)
plt.show()

print("Calibration curve saved to:", FIGURES_DIR / "response_baseline_calibration.png")

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title('Logistic Regression Confusion Matrix', fontsize=14)
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)

sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=True)
axes[1].set_title('XGBoost Confusion Matrix', fontsize=14)
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "response_baseline_confusion_matrices.png", dpi=150)
plt.show()

print("Confusion matrices saved to:", FIGURES_DIR / "response_baseline_confusion_matrices.png")

### Save predictions

In [ ]:
predictions_df = pd.DataFrame({
    'y_true': y_test,
    'logistic_proba': y_pred_proba_lr,
    'logistic_pred': y_pred_lr,
    'xgboost_proba': y_pred_proba_xgb,
    'xgboost_pred': y_pred_xgb
})

predictions_path = PREDICTIONS_DIR / "response_baseline_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)

print("Predictions saved to:", predictions_path)
print("\nPredictions sample:")
display(predictions_df.head(10))

### Save results

In [ ]:
results = {
    'logistic_regression': {
        'roc_auc': float(lr_auc),
        'precision': float(lr_precision),
        'recall': float(lr_recall),
        'f1_score': float(lr_f1),
        'brier_score': float(lr_brier),
        'confusion_matrix': cm_lr.tolist()
    },
    'xgboost': {
        'roc_auc': float(xgb_auc),
        'precision': float(xgb_precision),
        'recall': float(xgb_recall),
        'f1_score': float(xgb_f1),
        'brier_score': float(xgb_brier),
        'confusion_matrix': cm_xgb.tolist()
    },
    'metadata': {
        'outcome': PRIMARY_OUTCOME,
        'test_size': int(len(y_test)),
        'test_visit_rate': float(y_test.mean()),
        'model_type': 'response_baseline'
    }
}

results_path = MODEL_RESULTS_DIR / "response_baseline_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=4)

print("Model results saved to:", results_path)

### Summary

This notebook trained logistic regression and XGBoost models to predict customer visits. Both models achieved reasonable discrimination (ROC AUC ~0.55-0.60) but predict who will visit rather than who will visit *because* of the email. Traditional response models may recommend targeting customers who would have visited anyway, wasting resources. The predictions are saved for comparison with uplift models that explicitly estimate treatment effects.